# Optional Decision Tree Classifier

This notebook compares a simple decision tree classifier against the logistic regression baseline. The decision tree is optional because event datasets can be small, and trees can overfit sparse residual-event samples. The tree should only be used when there are enough labeled events and enough examples in both the success and failure classes.

## Research setup

The target remains the event label defined during labeling:

`1 = residual reverts before stop-loss`

`0 = stop-loss hits first or no reversion inside the holding window`

The tree uses the same event-level features known at signal time. It does not use future residual paths, exit dates, realized returns after entry, or label-derived information as features.

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd

from src.logistic_model import LogisticConfig, fit_logistic_regression, make_prediction_frame
from src.metrics import evaluate_predictions
from src.tree_model import (
    DecisionTreeConfig,
    compare_tree_to_logistic,
    dataset_large_enough,
    feature_split_summary,
    fit_decision_tree,
    leaf_summary,
    make_tree_prediction_frame,
    prepare_tree_frame,
)

DATA_DIR = Path('../data/processed')
FIG_DIR = Path('../figures')
DATA_DIR.mkdir(parents=True, exist_ok=True)
FIG_DIR.mkdir(parents=True, exist_ok=True)

In [ ]:
feature_path = DATA_DIR / 'event_feature_matrix.csv'

if feature_path.exists():
    events = pd.read_csv(feature_path, parse_dates=['event_date'])
else:
    events = pd.DataFrame()

if events.empty or not dataset_large_enough(events, min_events=100, min_positive=20, min_negative=20):
    rng = np.random.default_rng(14)
    n = 180
    dates = pd.date_range('2022-01-03', periods=n, freq='B')
    residual_z_score = rng.normal(0.0, 1.0, n)
    residual_volatility = rng.lognormal(mean=-3.7, sigma=0.35, size=n)
    beta_stability = rng.lognormal(mean=-4.1, sigma=0.40, size=n)
    market_return = rng.normal(0.0002, 0.012, n)
    recent_drawdown = -np.abs(rng.normal(0.035, 0.025, n))
    nonlinear_score = (
        1.2 * (np.abs(residual_z_score) > 0.9).astype(float)
        - 1.0 * (residual_volatility > np.quantile(residual_volatility, 0.72)).astype(float)
        - 0.8 * (beta_stability > np.quantile(beta_stability, 0.70)).astype(float)
        + 0.7 * ((market_return > -0.004) & (recent_drawdown > -0.06)).astype(float)
        + rng.normal(0.0, 0.45, n)
    )
    probability = 1.0 / (1.0 + np.exp(-nonlinear_score))
    label = (rng.random(n) < probability).astype(int)
    events = pd.DataFrame({
        'event_id': [f'optional_tree_evt_{i:04d}' for i in range(n)],
        'triplet_id': np.where(np.arange(n) % 2 == 0, 'NVDA_SMH_QQQ', 'AAPL_QQQ_XLK'),
        'method': 'kalman_random_walk',
        'event_date': dates,
        'residual_z_score': residual_z_score,
        'residual_volatility': residual_volatility,
        'beta_stability': beta_stability,
        'market_return': market_return,
        'recent_drawdown': recent_drawdown,
        'label': label,
    })
    print('Using synthetic placeholder events because the available event matrix is too small for the optional decision tree.')
else:
    print('Using project event feature matrix.')

events.shape

## Feature selection

The tree is limited to numeric event-time features. Metadata, outcomes, exit reasons, holding period outcomes, and label-derived columns are excluded.

In [ ]:
blocked = {
    'label', 'outcome', 'exit_reason', 'exit_date', 'holding_period',
    'event_id', 'triplet_id', 'method', 'event_date',
    'target_symbol', 'hedge_symbol_1', 'hedge_symbol_2', 'side',
}
feature_columns = [
    col for col in events.columns
    if col not in blocked and pd.api.types.is_numeric_dtype(events[col])
]

preferred = [
    'residual_z_score', 'residual_volatility', 'residual_autocorrelation',
    'half_life_estimate', 'rolling_r_squared', 'beta_stability',
    'correlation_stability', 'target_return_volatility', 'market_return',
    'sector_return', 'recent_drawdown', 'distance_from_moving_average',
]
feature_columns = [col for col in preferred if col in feature_columns] or feature_columns[:8]
prepared, feature_columns = prepare_tree_frame(events, feature_columns=feature_columns)
feature_columns

In [ ]:
prepared = prepared.sort_values('event_date').reset_index(drop=True)
split_idx = int(len(prepared) * 0.70)
train = prepared.iloc[:split_idx].copy()
validation = prepared.iloc[split_idx:].copy()

config = DecisionTreeConfig(
    max_depth=3,
    min_samples_split=24,
    min_samples_leaf=10,
    criterion='gini',
    threshold=0.5,
)

tree = fit_decision_tree(train[feature_columns], train['label'], config=config)
tree_predictions = make_tree_prediction_frame(validation, tree, feature_columns, split_name='validation')
tree_splits = feature_split_summary(tree)
tree_leaves = leaf_summary(tree)

tree_predictions.to_csv(DATA_DIR / 'decision_tree_predictions.csv', index=False)
tree_splits.to_csv(DATA_DIR / 'decision_tree_feature_split_summary.csv', index=False)
tree_leaves.to_csv(DATA_DIR / 'decision_tree_leaf_summary.csv', index=False)

tree_splits

## Compare against logistic regression

This comparison is not meant to prove the tree is better. It checks whether the optional nonlinear model adds useful validation performance relative to the simpler logistic regression baseline.

In [ ]:
logistic = fit_logistic_regression(
    train[feature_columns],
    train['label'],
    config=LogisticConfig(learning_rate=0.08, max_iter=1000, l2_penalty=0.5),
)
logistic_predictions = make_prediction_frame(validation, logistic, feature_columns, split_name='validation')
comparison = compare_tree_to_logistic(tree_predictions, logistic_predictions, threshold=0.5)
comparison.to_csv(DATA_DIR / 'decision_tree_vs_logistic_comparison.csv', index=False)
comparison

In [ ]:
tree_eval = evaluate_predictions(tree_predictions, threshold=0.5, n_bins=5)
tree_eval['model_evaluation_summary']

## Interpretation

The feature split summary shows which variables the tree used and how much impurity reduction each split created. Deep trees or tiny leaf nodes are not appropriate for this project because the event dataset can be small. If the decision tree only performs well in sample, or if the selected splits are unstable across walk-forward folds, keep logistic regression as the main model.